### Importing important Libraries

In [ ]:
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import re

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import cross_val_score
from sklearn.metrics import RocCurveDisplay, roc_curve, auc

from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from IPython.display import display

from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, roc_auc_score, roc_curve


In [5]:
#read dataset
Dataset = pd.read_csv('../data/train.csv')

### Load Features

#### Load handcrafted features

In [115]:
X_train = pd.read_csv('../features/train_features.csv')
X_test = pd.read_csv('../features/test_features.csv')

In [116]:
labels = ['toxic','severe_toxic','obscene','threat','insult','identity_hate']

y_train = X_train[labels]
y_test = X_test[labels]

In [117]:
drop_cols = [
'comment_text',
'preprocessed_text',
'toxic','severe_toxic','obscene',
'threat','insult','identity_hate',
'any_label'
]

X_train_features = X_train.drop(columns=drop_cols)
X_test_features = X_test.drop(columns=drop_cols)

print(X_train_features.shape)

(127656, 20)


Handcrafted numerical features have different ranges, which can bias distance-based and gradient-based models. Therefore, we apply StandardScaler to normalize these features. TF-IDF and embedding features are not scaled because they are already normalized or structured representations of text.

In [29]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_features)
X_test_scaled = scaler.transform(X_test_features)

In [30]:
print("Train handcrafted feature shape:", X_train_scaled.shape)
print("Test handcrafted feature shape:", X_test_scaled.shape)

Train handcrafted feature shape: (127656, 20)
Test handcrafted feature shape: (31915, 20)


In [31]:
print('Y_train shape:', y_train.shape)
print('Y_test shape:', y_test.shape)

Y_train shape: (127656, 6)
Y_test shape: (31915, 6)


#### Load Tfidf Features

In [32]:
from scipy import sparse

X_train_tfidf = sparse.load_npz("../features/X_train_tfidf.npz")
X_test_tfidf = sparse.load_npz("../features/X_test_tfidf.npz")

In [33]:
print("Train TF-IDF shape:", X_train_tfidf.shape)
print("Test TF-IDF shape:", X_test_tfidf.shape)

Train TF-IDF shape: (127656, 5000)
Test TF-IDF shape: (31915, 5000)


#### Load Word2vec Features

In [34]:
X_train_word2vec = np.load('../features/X_train_w2v.npy')
X_test_word2vec  = np.load('../features/X_test_w2v.npy')

In [35]:
print('Train word2vec shape:', X_train_word2vec.shape)
print('Test word2vec shape:', X_test_word2vec.shape)

Train word2vec shape: (127656, 100)
Test word2vec shape: (31915, 100)


#### Load Glove features

In [36]:
X_train_glove = np.load('../features/X_train_glove.npy')
X_test_glove  = np.load('../features/X_test_glove.npy')

In [37]:
print('Train glove shape:', X_train_glove.shape)
print('Test glove shape:', X_test_glove.shape)

Train glove shape: (127656, 100)
Test glove shape: (31915, 100)


# Baseline Models

## Evaluation Metrics Function

In [65]:
from IPython.display import display

def evaluate_model(y_true, y_pred, y_proba, labels):

    results = {}

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    y_proba = np.asarray(y_proba)

    # Global metrics
    for avg in ["micro","macro"]:
        results[f'precision_{avg}'] = precision_score(y_true, y_pred, average=avg, zero_division=0)
        results[f'recall_{avg}']    = recall_score(y_true, y_pred, average=avg, zero_division=0)
        results[f'f1_{avg}']        = f1_score(y_true, y_pred, average=avg, zero_division=0)

    results['roc_auc_macro'] = roc_auc_score(y_true, y_proba, average="macro")
    results['roc_auc_micro'] = roc_auc_score(y_true, y_proba, average="micro")

    print("\n--- Global Metrics ---")
    for k,v in results.items():
        print(f"{k}: {v:.4f}")

    rows = []
    for i,label in enumerate(labels):

        prec = precision_score(y_true[:,i], y_pred[:,i], zero_division=0)
        rec  = recall_score(y_true[:,i], y_pred[:,i], zero_division=0)
        f1   = f1_score(y_true[:,i], y_pred[:,i], zero_division=0)

        auc = None
        if len(set(y_true[:,i])) > 1:
            auc = roc_auc_score(y_true[:,i], y_proba[:,i])

        rows.append([label,prec,rec,f1,auc])

    per_label_df = pd.DataFrame(rows, columns=["Label","Precision","Recall","F1","ROC-AUC"])

    print("\n--- Per-label Metrics ---")
    display(per_label_df)

    return results, per_label_df

## Logistic Regression

To establish an initial performance benchmark, we begin with Logistic Regression as our baseline model. Logistic Regression is widely used for text classification tasks because it performs well with high-dimensional sparse features such as TF-IDF vectors. Additionally, it is computationally efficient and easy to interpret. Since the toxic comment classification task is a multi-label problem, we employ the One-Vs-Rest strategy, which trains one binary classifier for each label.

### Logistic Regression on TFDIF

In [68]:
ovr_lr_tfidf = OneVsRestClassifier(LogisticRegression(max_iter=200, random_state=42))

In [69]:
print("Training Logistic Regression (TF-IDF)...")
ovr_lr_tfidf.fit(X_train_tfidf, y_train)

# Predictions
y_pred_lr_tfidf = ovr_lr_tfidf.predict(X_test_tfidf)
y_pred_proba_lr_tfidf = ovr_lr_tfidf.predict_proba(X_test_tfidf)

Training Logistic Regression (TF-IDF)...


In [70]:
print('Evaluation of Logistic Regression model on tfidf')
results_lr_tfidf, table_lr_tfidf = evaluate_model(y_test, y_pred_lr_tfidf, y_pred_proba_lr_tfidf, labels);

Evaluation of Logistic Regression model on tfidf

--- Global Metrics ---
precision_micro: 0.8554
recall_micro: 0.5723
f1_micro: 0.6858
precision_macro: 0.7652
recall_macro: 0.3986
f1_macro: 0.4992
roc_auc_macro: 0.9731
roc_auc_micro: 0.9790

--- Per-label Metrics ---


,Label,Precision,Recall,F1,ROC-AUC
0,toxic,0.889098,0.621142,0.731349,0.964169
1,severe_toxic,0.531469,0.249180,0.339286,0.983584
2,obscene,0.893008,0.637786,0.744121,0.978002
3,threat,0.785714,0.112245,0.196429,0.972526
4,insult,0.804566,0.559010,0.659678,0.974883
5,identity_hate,0.687500,0.212355,0.324484,0.965635


### Logistic regression on Handcrafted features

In [71]:
ovr_lr_hand = OneVsRestClassifier(LogisticRegression(max_iter=200, random_state=42))

print("Training Logistic Regression (handcrafted features)...")
ovr_lr_hand.fit(X_train_scaled, y_train)

# Predictions
y_pred_lr_hand = ovr_lr_hand.predict(X_test_scaled)
y_pred_proba_lr_hand = ovr_lr_hand.predict_proba(X_test_scaled)

Training Logistic Regression (handcrafted features)...


In [72]:
print('Evaluation of Logistic Regression model on handcrafted features')
results_lr_hand, table_lr_hand = evaluate_model(y_test, y_pred_lr_hand, y_pred_proba_lr_hand, labels);

Evaluation of Logistic Regression model on handcrafted features

--- Global Metrics ---
precision_micro: 0.7895
recall_micro: 0.3149
f1_micro: 0.4502
precision_macro: 0.5909
recall_macro: 0.2062
f1_macro: 0.2962
roc_auc_macro: 0.8589
roc_auc_micro: 0.9040

--- Per-label Metrics ---


,Label,Precision,Recall,F1,ROC-AUC
0,toxic,0.837635,0.330269,0.473746,0.823442
1,severe_toxic,0.373737,0.121311,0.183168,0.907690
2,obscene,0.829439,0.427196,0.563940,0.870191
3,threat,0.454545,0.051020,0.091743,0.849764
4,insult,0.735915,0.265228,0.389925,0.858379
5,identity_hate,0.314286,0.042471,0.074830,0.844128


### Logistic Regression on word2vec

In [73]:
ovr_lr_w2v = OneVsRestClassifier(LogisticRegression(max_iter=200, random_state=42))

print("Training Logistic Regression (Word2Vec)...")
ovr_lr_w2v.fit(X_train_word2vec, y_train)

# Predictions
y_pred_lr_w2v = ovr_lr_w2v.predict(X_test_word2vec)
y_pred_proba_lr_w2v = ovr_lr_w2v.predict_proba(X_test_word2vec)

Training Logistic Regression (Word2Vec)...


In [74]:
print('Evaluation of Logistic Regression model on Word2vec features')
results_lr_w2v, table_lr_w2v = evaluate_model(y_test, y_pred_lr_w2v, y_pred_proba_lr_w2v, labels);

Evaluation of Logistic Regression model on Word2vec features

--- Global Metrics ---
precision_micro: 0.7329
recall_micro: 0.4234
f1_micro: 0.5367
precision_macro: 0.5951
recall_macro: 0.2760
f1_macro: 0.3593
roc_auc_macro: 0.9438
roc_auc_micro: 0.9590

--- Per-label Metrics ---


,Label,Precision,Recall,F1,ROC-AUC
0,toxic,0.768523,0.493762,0.601239,0.936287
1,severe_toxic,0.443662,0.206557,0.281879,0.969828
2,obscene,0.752239,0.454874,0.566929,0.945756
3,threat,0.636364,0.071429,0.128440,0.929050
4,insult,0.704358,0.379442,0.493196,0.949791
5,identity_hate,0.265306,0.050193,0.084416,0.932261


### Logistic Regression on Glove features

In [ ]:
ovr_lr_glove = OneVsRestClassifier(LogisticRegression(max_iter=200, random_state=42))

print("Training Logistic Regression (Glove)...")
ovr_lr_glove.fit(X_train_glove, y_train)

# Predictions
y_pred__lr_glove = ovr_lr_glove.predict(X_test_glove)
y_pred_proba_lr_glove = ovr_lr_glove.predict_proba(X_test_glove)

Training Logistic Regression (Glove)...


In [ ]:
print('Evaluation of Logistic Regression model on glove features')
results_lr_glove, table_lr_glove = evaluate_model(y_test, y_pred__lr_glove, y_pred_proba_lr_glove, labels);

Evaluation of Logistic Regression model on glove features

--- Global Metrics ---
precision_micro: 0.6598
recall_micro: 0.2803
f1_micro: 0.3935
precision_macro: 0.5010
recall_macro: 0.1731
f1_macro: 0.2481
roc_auc_macro: 0.9154
roc_auc_micro: 0.9359

--- Per-label Metrics ---


,Label,Precision,Recall,F1,ROC-AUC
0,toxic,0.697817,0.335850,0.453457,0.899275
1,severe_toxic,0.350000,0.068852,0.115068,0.942889
2,obscene,0.668005,0.300241,0.414280,0.909767
3,threat,0.250000,0.020408,0.037736,0.898960
4,insult,0.611465,0.243655,0.348457,0.916333
5,identity_hate,0.428571,0.069498,0.119601,0.925333


In [78]:
comparison_lr = pd.DataFrame({
    "TFIDF": results_lr_tfidf,
    "Handcrafted": results_lr_hand,
    "Word2Vec": results_lr_w2v,
    "GloVe": results_lr_glove
})

comparison_lr.T

,precision_micro,recall_micro,f1_micro,precision_macro,recall_macro,f1_macro,roc_auc_macro,roc_auc_micro
TFIDF,0.855391,0.572272,0.685759,0.765226,0.398620,0.499224,0.973133,0.979026
Handcrafted,0.789531,0.314857,0.450185,0.590926,0.206249,0.296225,0.858932,0.903960
Word2Vec,0.732868,0.423409,0.536728,0.595075,0.276043,0.359350,0.943829,0.958959
GloVe,0.659776,0.280305,0.393453,0.500976,0.173084,0.248100,0.915426,0.935934


The results indicate that TF-IDF features provide the best performance across all evaluation metrics, achieving the highest ROC-AUC, F1 score, and recall. This suggests that toxicity detection relies heavily on specific lexical patterns and offensive keywords, which TF-IDF captures effectively. Word embeddings such as Word2Vec and GloVe show moderate performance by capturing semantic relationships but slightly underperform compared to TF-IDF. Handcrafted features perform the worst, indicating that simple structural characteristics of comments are insufficient to accurately detect toxic language.

In [80]:
table_lr_tfidf

,Label,Precision,Recall,F1,ROC-AUC
0,toxic,0.889098,0.621142,0.731349,0.964169
1,severe_toxic,0.531469,0.249180,0.339286,0.983584
2,obscene,0.893008,0.637786,0.744121,0.978002
3,threat,0.785714,0.112245,0.196429,0.972526
4,insult,0.804566,0.559010,0.659678,0.974883
5,identity_hate,0.687500,0.212355,0.324484,0.965635


The label-wise evaluation shows that the model performs best on the more frequent categories such as toxic, obscene, and insult, achieving F1 scores above 0.65. In contrast, performance is significantly lower for minority classes such as threat, identity_hate, and severe_toxic, primarily due to their limited representation in the dataset.

## Multinomial Naive Bayes (TF-IDF)

In [81]:
ovr_nb = OneVsRestClassifier(MultinomialNB())

In [82]:
print("Training Naive Bayes (TF-IDF)...")
ovr_nb.fit(X_train_tfidf, y_train)

# Predictions
y_pred_nb = ovr_nb.predict(X_test_tfidf)
y_pred_proba_nb = ovr_nb.predict_proba(X_test_tfidf)

Training Naive Bayes (TF-IDF)...


In [84]:
print('Evaluation of Naive Bayes model')
results_nb_tfidf, table_nb_tfidf = evaluate_model(y_test, y_pred_nb, y_pred_proba_nb, labels)

Evaluation of Naive Bayes model

--- Global Metrics ---
precision_micro: 0.8465
recall_micro: 0.4898
f1_micro: 0.6205
precision_macro: 0.5888
recall_macro: 0.3269
f1_macro: 0.4146
roc_auc_macro: 0.9570
roc_auc_micro: 0.9682

--- Per-label Metrics ---


,Label,Precision,Recall,F1,ROC-AUC
0,toxic,0.909039,0.524951,0.665557,0.952022
1,severe_toxic,0.514451,0.291803,0.372385,0.975599
2,obscene,0.873106,0.554753,0.678440,0.962932
3,threat,0.000000,0.000000,0.000000,0.939009
4,insult,0.793568,0.485406,0.602362,0.963335
5,identity_hate,0.442623,0.104247,0.168750,0.948952


# Complex Models

## Linear Support Vector Machine (TF-IDF)

In [89]:
ovr_svm = OneVsRestClassifier(LinearSVC())

print("Training Linear SVM (TF-IDF)...")
ovr_svm.fit(X_train_tfidf, y_train)

# Predictions
y_pred_svm = ovr_svm.predict(X_test_tfidf)

# LinearSVC doesn't output probabilities
y_proba_svm = y_pred_svm

Training Linear SVM (TF-IDF)...


In [90]:
print("Evaluation of Linear SVM model")
results_svm_tfidf, table_svm_tfidf = evaluate_model(
    y_test,
    y_pred_svm,
    y_proba_svm,
    labels)

Evaluation of Linear SVM model

--- Global Metrics ---
precision_micro: 0.8330
recall_micro: 0.6183
f1_micro: 0.7098
precision_macro: 0.7206
recall_macro: 0.4448
f1_macro: 0.5356
roc_auc_macro: 0.7200
roc_auc_micro: 0.8068

--- Per-label Metrics ---


,Label,Precision,Recall,F1,ROC-AUC
0,toxic,0.862303,0.666120,0.751621,0.827448
1,severe_toxic,0.531915,0.245902,0.336323,0.621907
2,obscene,0.872067,0.693141,0.772377,0.843777
3,threat,0.629630,0.173469,0.272000,0.586578
4,insult,0.791806,0.600888,0.683261,0.796341
5,identity_hate,0.635593,0.289575,0.397878,0.644108


# Tree models - RF, DT

## Random Forest (TF-IDF)

In [91]:
ovr_rfc = OneVsRestClassifier(RandomForestClassifier(random_state = 42))

In [92]:
print("Training Random forest (TF-IDF)...")
ovr_rfc.fit(X_train_tfidf, y_train)

# Predictions
y_pred_rfc = ovr_rfc.predict(X_test_tfidf)
y_pred_proba_rfc = ovr_rfc.predict_proba(X_test_tfidf)

Training Random forest (TF-IDF)...


In [93]:
print("Evaluation of Random Forest model")
results_rf_tfidf, table_rf_tfidf = evaluate_model(
    y_test,
    y_pred_rfc,
    y_pred_proba_rfc,
    labels
)

Evaluation of Random Forest model

--- Global Metrics ---
precision_micro: 0.8204
recall_micro: 0.5926
f1_micro: 0.6881
precision_macro: 0.7062
recall_macro: 0.3769
f1_macro: 0.4507
roc_auc_macro: 0.9479
roc_auc_micro: 0.9673

--- Per-label Metrics ---


,Label,Precision,Recall,F1,ROC-AUC
0,toxic,0.847179,0.626067,0.720030,0.946999
1,severe_toxic,0.465753,0.111475,0.179894,0.970541
2,obscene,0.843552,0.720217,0.777020,0.969590
3,threat,0.666667,0.061224,0.112150,0.907650
4,insult,0.774917,0.592005,0.671223,0.961575
5,identity_hate,0.639344,0.150579,0.243750,0.931207


## MLP Classifier (TF-IDF)


In [94]:
ovr_mlp = OneVsRestClassifier(
    MLPClassifier(
        hidden_layer_sizes=(100,),
        max_iter=20,
        random_state=42
    )
)

In [95]:
print("Training MLP (TF-IDF)...")
ovr_mlp.fit(X_train_tfidf, y_train)

y_pred_mlp = ovr_mlp.predict(X_test_tfidf)
y_pred_proba_mlp = ovr_mlp.predict_proba(X_test_tfidf)

Training MLP (TF-IDF)...


/Users/nehayadav/Desktop/Ml-Projects/toxic-comment-classifier/toxic_env/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/nehayadav/Desktop/Ml-Projects/toxic-comment-classifier/toxic_env/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/nehayadav/Desktop/Ml-Projects/toxic-comment-classifier/toxic_env/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/nehayadav/Desktop/Ml-Projects/toxic-comment-classifier/toxic_env/lib/python3.11/site-packages/sklearn/neural_network/_multilay

In [96]:
print("Evaluation of MLP model")
results_mlp_tfidf, table_mlp_tfidf = evaluate_model(
    y_test,
    y_pred_mlp,
    y_pred_proba_mlp,
    labels
)

Evaluation of MLP model

--- Global Metrics ---
precision_micro: 0.7307
recall_micro: 0.6548
f1_micro: 0.6907
precision_macro: 0.5934
recall_macro: 0.5080
f1_macro: 0.5451
roc_auc_macro: 0.9534
roc_auc_micro: 0.9685

--- Per-label Metrics ---


,Label,Precision,Recall,F1,ROC-AUC
0,toxic,0.776782,0.683191,0.726987,0.947858
1,severe_toxic,0.390244,0.367213,0.378378,0.967265
2,obscene,0.782723,0.719615,0.749843,0.962316
3,threat,0.428571,0.275510,0.335404,0.941290
4,insult,0.703978,0.662437,0.682576,0.960084
5,identity_hate,0.478261,0.339768,0.397291,0.941685


## Conclusion

In [105]:
model_comparison = pd.DataFrame({
    "Logistic Regression": results_lr_tfidf,
    "Naive Bayes": results_nb_tfidf,
    "Linear SVM": results_svm_tfidf,
    "Random Forest": results_rf_tfidf,
    "MLP": results_mlp_tfidf
}).T

model_comparison.round(3)

,precision_micro,recall_micro,f1_micro,precision_macro,recall_macro,f1_macro,roc_auc_macro,roc_auc_micro
Logistic Regression,0.855,0.572,0.686,0.765,0.399,0.499,0.973,0.979
Naive Bayes,0.846,0.490,0.621,0.589,0.327,0.415,0.957,0.968
Linear SVM,0.833,0.618,0.710,0.721,0.445,0.536,0.720,0.807
Random Forest,0.820,0.593,0.688,0.706,0.377,0.451,0.948,0.967
MLP,0.731,0.655,0.691,0.593,0.508,0.545,0.953,0.969


Among the evaluated models, Linear SVM achieved the highest F1-score and recall values, indicating superior performance in detecting toxic comments. Logistic Regression achieved the highest ROC-AUC score, demonstrating strong discrimination capability; however, Linear SVM produced better overall classification performance. Tree-based models such as Random Forest and probabilistic models like Naive Bayes performed reasonably well but did not outperform the linear models. Therefore, Linear SVM with TF-IDF features was selected as the final model for this task.

### Label wise performance for FINAL MODEL - LinearSVM (TF-IDF)

In [98]:
table_svm_tfidf

,Label,Precision,Recall,F1,ROC-AUC
0,toxic,0.862303,0.666120,0.751621,0.827448
1,severe_toxic,0.531915,0.245902,0.336323,0.621907
2,obscene,0.872067,0.693141,0.772377,0.843777
3,threat,0.629630,0.173469,0.272000,0.586578
4,insult,0.791806,0.600888,0.683261,0.796341
5,identity_hate,0.635593,0.289575,0.397878,0.644108


Logistic Regression achieved the highest ROC-AUC scores due to its probabilistic outputs, which allow better ranking of predictions. Linear SVM does not directly produce probability estimates, leading to lower ROC-AUC values. However, Linear SVM achieved higher F1 and recall scores, indicating better overall classification performance for detecting toxic comments.

## Hyperparameter Tuning – Linear SVM

In [112]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import LinearSVC

svm_model = OneVsRestClassifier(LinearSVC())

param_grid = {
    "estimator__C": [0.01, 0.1, 1, 10],
    "estimator__class_weight": [None, "balanced"]
}

grid_search = GridSearchCV(
    svm_model,
    param_grid,
    scoring="f1_micro",
    cv=3,
    n_jobs=-1
)

print("Running GridSearchCV for Linear SVM...")
grid_search.fit(X_train_tfidf, y_train)

best_svm = grid_search.best_estimator_

print("Best Parameters:", grid_search.best_params_)

Running GridSearchCV for Linear SVM...


/Users/nehayadav/Desktop/Ml-Projects/toxic-comment-classifier/toxic_env/lib/python3.11/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/nehayadav/Desktop/Ml-Projects/toxic-comment-classifier/toxic_env/lib/python3.11/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/nehayadav/Desktop/Ml-Projects/toxic-comment-classifier/toxic_env/lib/python3.11/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Best Parameters: {'estimator__C': 1, 'estimator__class_weight': None}


Hyperparameter tuning was performed for the Linear SVM classifier by testing multiple values of the regularization parameter C. The results showed that C = 1, which is the default value, provided the best performance. This indicates that the baseline configuration of Linear SVM was already well suited for this dataset.

In [113]:
y_pred_svm_tuned = best_svm.predict(X_test_tfidf)

# still no probabilities
y_proba_svm_tuned = y_pred_svm_tuned

print("Evaluation of Tuned Linear SVM")
results_svm_tuned, table_svm_tuned = evaluate_model(
    y_test,
    y_pred_svm_tuned,
    y_proba_svm_tuned,
    labels
)

Evaluation of Tuned Linear SVM

--- Global Metrics ---
precision_micro: 0.8330
recall_micro: 0.6183
f1_micro: 0.7098
precision_macro: 0.7206
recall_macro: 0.4448
f1_macro: 0.5356
roc_auc_macro: 0.7200
roc_auc_micro: 0.8068

--- Per-label Metrics ---


,Label,Precision,Recall,F1,ROC-AUC
0,toxic,0.862303,0.666120,0.751621,0.827448
1,severe_toxic,0.531915,0.245902,0.336323,0.621907
2,obscene,0.872067,0.693141,0.772377,0.843777
3,threat,0.629630,0.173469,0.272000,0.586578
4,insult,0.791806,0.600888,0.683261,0.796341
5,identity_hate,0.635593,0.289575,0.397878,0.644108


Hyperparameter tuning was performed for the Linear SVM classifier using GridSearchCV. The regularization parameter C and the class_weight parameter were tuned to improve model performance and address the class imbalance present in the dataset.

# Error-Analysis

In [118]:
errors = pd.DataFrame({
    "comment_text": X_test["comment_text"],
    "true_label": y_test["toxic"],
    "predicted_label": y_pred_svm[:,0]
})

misclassified = errors[errors["true_label"] != errors["predicted_label"]]
misclassified.head(10)

,comment_text,true_label,predicted_label
20,"""\nHe's also, to be fair, managed to use other...",1,0
21,Parasite Pig \n\nWith the recent death of the ...,0,1
25,"Oh so andy can have a history of this crap, ye...",1,0
36,Oppose Iset? Crazy.,1,0
70,(unless they relate to my hatred of Arabs),1,0
93,I don't care what you say here. I don't believ...,1,0
99,Boring\n\nWhy are you so boring Gail.. stop ha...,1,0
108,"LOTHAR VON TROTHA\nGOOD RIDDANCE TO HIM, HE KI...",1,0
110,You are really a dog .... this is the photo of...,1,0
111,"""\n\n personal attack \n\ndear toddst1,\n\nple...",1,0


### Error Analysis Observations

The misclassified examples reveal that many errors occur when toxicity is expressed indirectly or through sarcasm. Some comments contain offensive words used in a contextual or quoted manner, which leads to false positive predictions. Additionally, minority classes such as threat and identity_hate are often harder to detect due to their limited representation in the training data.

## Future Work

Future improvements could include experimenting with contextual embeddings such as BERT or RoBERTa, applying advanced techniques for handling class imbalance, and exploring deep learning architectures for improved toxic comment detection.